# PDNC Hackathon - Challenge 01: Aspect-Based Sentiment Analysis
Pipeline: data -> aspect-conditioned tokenization -> transformer + 3-class head -> validate (Macro F1) -> predict -> submission.csv

**Before running:** set `TRAIN_PATH` / `TEST_PATH` in cell 2 and check the column names printed there.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn sentencepiece tiktoken

In [ ]:
import os, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

# ---- CONFIG: edit these ----
import glob
def find(name):   # finds the file wherever Kaggle/Colab mounted it
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"**/{name}", recursive=True) + glob.glob(f"/content/**/{name}", recursive=True)
    return hits[0]
TRAIN_PATH, TEST_PATH = find("train.csv"), find("test.csv")
USE_TAGGED = False   # True = use context_with_aspect_tag ($T$ slot) instead of raw text
MODEL_NAME = "microsoft/deberta-v3-base"   # alternatives: roberta-base, bert-base-uncased
MAX_LEN, BATCH, LR, EPOCHS, SEED = 128, 16, 2e-5, 4, 42

# keep_default_na=False: one training aspect is the word "nan" and pandas would turn it into a missing value
train_df = pd.read_csv(TRAIN_PATH, keep_default_na=False)
test_df  = pd.read_csv(TEST_PATH,  keep_default_na=False)
print(train_df.shape, test_df.shape)
print(train_df.columns.tolist())
train_df.head()

## 1. Explore: labels, lengths, aspects

In [ ]:
TEXT_COL = "context_with_aspect_tag" if USE_TAGGED else "text"
ASPECT_COL, LABEL_COL = "aspect_term", "label"
print("Input text column:", TEXT_COL)
print(train_df[LABEL_COL].value_counts())
print("Avg review words:", train_df[TEXT_COL].str.split().str.len().describe())
g = train_df.groupby("text")["label"].nunique()
print("Reviews with >1 aspect:", (train_df.groupby("text").size() > 1).sum(), "of", train_df.text.nunique())
print("Reviews whose aspects have DIFFERENT sentiments:", (g > 1).sum())

## 2. Label mapping + stratified validation split

In [ ]:
LABELS = ["Negative", "Neutral", "Positive"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

train_df["y"] = train_df[LABEL_COL].str.strip().str.capitalize().map(label2id)
assert train_df["y"].notna().all(), "Unmapped labels - inspect unique values"

tr, va = train_test_split(train_df, test_size=0.1, stratify=train_df["y"], random_state=SEED)
print(len(tr), len(va))

## 2b. Quick baselines (for the 'baseline vs ours' slide)
No GPU needed. Expected: review-only ~0.38 Macro F1 (near chance), aspect-aware ~0.67.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack
def near(r, k=4):
    w = r.context_with_aspect_tag.split(); i = next((j for j, t in enumerate(w) if "$T$" in t), 0)
    return " ".join(w[max(0, i-k):i] + w[i+1:i+1+k])
a, b = tr.copy(), va.copy()
a["near"], b["near"] = a.apply(near, axis=1), b.apply(near, axis=1)
v1 = TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True)
c1 = LogisticRegression(max_iter=2000, class_weight="balanced").fit(v1.fit_transform(a.text), a.y)
print("Baseline A (review only, ignores aspect) Macro F1:", round(f1_score(b.y, c1.predict(v1.transform(b.text)), average="macro"), 4))
v2, v3 = TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True), TfidfVectorizer()
Xa = hstack([v1.transform(a.text), v2.fit_transform(a.near), v3.fit_transform(a.aspect_term)]).tocsr()
Xb = hstack([v1.transform(b.text), v2.transform(b.near), v3.transform(b.aspect_term)]).tocsr()
c2 = LogisticRegression(max_iter=3000, class_weight="balanced").fit(Xa, a.y)
print("Baseline B (aspect + nearby words) Macro F1:", round(f1_score(b.y, c2.predict(Xb), average="macro"), 4))

## 3. Tokenize as a sentence pair: [CLS] aspect [SEP] review [SEP]

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def enc(batch):
    return tok(batch[ASPECT_COL], batch[TEXT_COL], truncation="only_second", max_length=MAX_LEN)

def to_ds(df, labeled=True):
    cols = [ASPECT_COL, TEXT_COL] + (["y"] if labeled else [])
    ds = Dataset.from_pandas(df[cols].reset_index(drop=True))
    ds = ds.map(enc, batched=True)
    if labeled: ds = ds.rename_column("y", "labels")
    return ds.remove_columns([c for c in [ASPECT_COL, TEXT_COL] if c in ds.column_names])

train_ds, val_ds, test_ds = to_ds(tr), to_ds(va), to_ds(test_df, labeled=False)
collator = DataCollatorWithPadding(tok)

## 4. Model, class-weighted loss, training

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id)

counts = tr["y"].value_counts().sort_index().values
class_weights = torch.tensor(len(tr) / (3 * counts), dtype=torch.float)
print("class weights:", class_weights)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro"),
            "accuracy": accuracy_score(p.label_ids, preds)}

args = TrainingArguments(
    output_dir="absa_out", learning_rate=LR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 2,
    weight_decay=0.01, warmup_ratio=0.1,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(), seed=SEED, report_to="none")

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=collator, tokenizer=tok, compute_metrics=compute_metrics)
trainer.train()

## 5. Validation report (per-class F1, confusion)

In [ ]:
from sklearn.metrics import confusion_matrix
out = trainer.predict(val_ds)
vp = out.predictions.argmax(-1)
print(classification_report(out.label_ids, vp, target_names=LABELS, digits=4))
print("Macro F1:", f1_score(out.label_ids, vp, average="macro"))
print(confusion_matrix(out.label_ids, vp))

## 6. Error analysis (useful for the 2-minute defense)

In [ ]:
va = va.reset_index(drop=True)
va["pred"] = [id2label[i] for i in vp]
va["gold"] = va["y"].map(id2label)
errs = va[va.pred != va.gold]
print(f"{len(errs)} errors / {len(va)}")
errs[[ASPECT_COL, TEXT_COL, "gold", "pred"]].head(15)

## 7. Predict test set and write submission.csv

In [ ]:
tp = trainer.predict(test_ds).predictions.argmax(-1)
sub = pd.DataFrame({"id": test_df["id"], "prediction": [id2label[i] for i in tp]})
sub.to_csv("submission.csv", index=False)
print(sub.shape, sub.prediction.value_counts().to_dict())
assert len(sub) == 2000 and sub.isna().sum().sum() == 0
print("Now put submission.csv in the submission/ folder and run: python utils/validator.py")
sub.head()

## Improvement ideas (if time allows)
- Swap in `roberta-large` / `deberta-v3-large` (needs more GPU memory; lower batch size)
- Use `context_with_aspect_tag` (aspect wrapped in markers) as the review input instead of raw `text`
- Average logits from 3 seeds or 5-fold CV
- Try a sentiment-pretrained backbone (e.g. `cardiffnlp/twitter-roberta-base-sentiment-latest`) for a warm start